In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
import chromadb

In [ ]:
def load_files(file_path):
    return PyPDFLoader(file_path).load()

In [ ]:
#Carrega os documentos - base de conhecimento
docs = []
docs.extend(load_files("cmp_arq.pdf"))
print(len(docs))
docs.extend(load_files("Energy-Efficient_CPUFPGA-Based_CNN_Architecture_for_Intrusion_Detection_Systems.pdf"))
print(len(docs))

In [ ]:
print(docs[0])
print(len(docs))

In [ ]:
print(f"{docs[100].page_content[:500]}\n")
#print(docs[0].metadata)

In [ ]:
#Realiza a divisão dos documentos em pequenas porções
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024, chunk_overlap=256, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)
all_splits_ids = []
for index, spi in enumerate(all_splits):
    spi.id = str(index+1)
    all_splits_ids.append(spi.id)
print(len(all_splits))

In [ ]:
print(all_splits[2])

In [ ]:
#Inicializa o modelo de embedding (transforma texto em vetor)
embeddings = OllamaEmbeddings(
    model="mxbai-embed-large",
)
#Caso queira criar um banco em memória, sem persistência
#vector_store = InMemoryVectorStore(embeddings)
#ids = vector_store.add_documents(documents=all_splits)

In [ ]:
#collection_name='cmp_arq_512_64_and_efficiency_paper_mxbai'
collection_name='cmp_arq_1024_256_and_efficiency_paper_mxbai'

In [ ]:
#Inicializa o banco de dados vetorial

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory='./db/'
)

persistent_client = chromadb.PersistentClient(path='./db/')
collection = persistent_client.get_or_create_collection(collection_name)
#collection.update(all_splits_ids, documents=all_splits_ids)

In [ ]:
collection.count()

In [ ]:
split_value = int(len(all_splits_ids)/2)
split_value

In [ ]:
ids = vector_store.add_documents(ids = all_splits_ids[:split_value], documents=all_splits[:split_value])
#ids = vector_store.update_documents(ids = all_splits_ids, documents=all_splits)

In [ ]:
ids = vector_store.add_documents(ids = all_splits_ids[split_value:], documents=all_splits[split_value:])
#ids = vector_store.update_documents(ids = all_splits_ids, documents=all_splits)

In [ ]:
#Inicializa o modelo de prompt (template)

prompt = ChatPromptTemplate.from_template("""
1. Assume that you are a computer engineer and architect specialist, use your knowledge to help build the given architecture.
2. You can use the context given to you for better understand the current architecture.
3. If you don't know what to answer, use "N/A".                                        
4. Replace ??? of the jsonObject below with possible components and connectivity details of 
a computer architecture for the given problem.
5. You can only manipulate the placeholders.
6. Don't modify any other value other than ??? 
7. Remove any other commentaries.

Question: {question} 
Context: {context} 
Answer:
""")
prompt

You must fill only the placeholders (?), keeping the rest of the structure intact, with the knowledge that you poses,
 with the missing components and connectivity details of a computer architecture of a FPGA and CPU that computes a CNN of 
 a intrusion detection system, using the following JSON structure and ensure the entire structure is returned fully filled.
 Note that are many placeholders. All of them must have a value. The answer must be the following architecture as json.


 Do not include explanations, comments, or additional text outside the JSON structure.


In [ ]:
#Montagem da pergunta
question = '''
Which are the components and connectivity that can replace the placeholders for a 
CPU+FPGA device that computes a CNN of a intrusion detection system (IDS) ?

Architecture:
{
    "components":[
        {"name":"Xeon", "components":[
                {"name":"QPI CTRL", "components":[]},
                {"name":"PCIe 0 CTRL", "components":[]},
                ???,
                {"name":"LLC", "components":[]},
        ]},
        {"name":"FPGA Arria 10", "components":[
            {"name":"FIU", "components":[
                {"name":"QPI CTRL", "components":[]},
                {{"name":"PCIe 0 CTRL", "components":[]},}
                ???,
                {"name":"Cache", "capacity":"64 kb"}
            ]
            },
        ]},
        {"name":"DRAM", "capacity":"64 GB"}
    ],
    "connections":[
        {"from": "Xeon", "to": "DRAM", "description": ""},
        {"to": "Xeon", "from": "DRAM", "description": ""},
        {"from": "Xeon.QPI CTRL", "to": "FPGA Arria 10.FIU.QPI CTRL", "description": "QPI (12,8 GB/s)"},
        {"from": "Xeon.PCIe 0 CTRL", "to": "FPGA Arria 10.FIU.PCIe 0 CTRL", "description": "PCIe 0 (16 GB/s)"},
        ???,
        {"from": "FPGA Arria 10.FIU.QPI CTRL", "to": "Xeon.QPI CTRL", "description": "QPI (12,8 GB/s)"},
        {"from": "FPGA Arria 10.FIU.PCIe 0 CTRL", "to": "Xeon.PCIe 0 CTRL", "description": "PCIe 0 (16 GB/s)"},
        ???,
        {"from": "Xeon.LLC", "to": "Xeon.QPI CTRL", "description": ""},
        {"to": "Xeon.LLC", "from": "Xeon.QPI CTRL", "description": ""},
        {"from": "Xeon.LLC", "to": "Xeon.PCIe 0 CTRL", "description": ""},
        ???,
        {"to": "Xeon.LLC", "from": "Xeon.PCIe 0 CTRL", "description": ""},
        ???,
        ???,
        ???,
        {"from": "FPGA Arria 10.FIU.QPI CTRL", "to": "FPGA Arria 10.FIU.Cache", "description": ""},
        {"to": "FPGA Arria 10.FIU.QPI CTRL", "from": "FPGA Arria 10.FIU.Cache", "description": ""},
        ???,
        ???,
    ]
}
'''

docs_content=[]

In [ ]:
#Busca documentos com similaridade com a questão, na base de conhecimentos, para servir como contexto da questão
retrieved_docs = vector_store.similarity_search_with_score(question, k=4) #default: 4

docs_content = ""
for doc, score in retrieved_docs:
    print(score, doc.page_content)
    #if score < 1:
    docs_content += doc.page_content
    #print(page_content)


In [ ]:
#Inicializa o modelo
llm = OllamaLLM(model="llama3.2:3b", verbose=True)
#llm = OllamaLLM(model="gemma3:12b", verbose=True)
#Monta a pergunta usando a questão e contexto no template
#cria um molde do input
input = {"question": question, "context": docs_content}
#Prepara as mensagens
messages = prompt.invoke(input)
#chain = prompt | llm
#chain.invoke(question)
#Realiza a chamada à LLM
answer = llm.invoke(messages)
print('answer: ', answer)
print('input:', input)

In [ ]:
#Pós-processamento para limpeza da resposta
import re
cleaned_response = re.sub(r'\s+', ' ', answer).strip()
#cleaned_response = answer.replace("\n", " ")
cleaned_response